# 1 Gate, highway, or route: one idea from three papers

The gate/highway/MoE family composes with any `(batch, hidden)` tensor — images, text, tabular data, anything. It sits here because
this lesson already has a window and a baseline table to beat, the same reason
`1_representation_and_head/04_search` put its search where a training run takes seconds: the
place makes the idea cheap to test, not because the idea belongs to one kind of data.

Two questions run through the notebook: does a mechanism earn its parameters against a model
with as many parameters and no mechanism, and what does a router do with the experts it has?


## 1.1 One idea, three resolutions

| component | mechanism | where it comes from |
|---|---|---|
| `Gate` | `sigmoid(Wx) * x` -- a learned per-dimension scale | the gate inside every LSTM cell (Hochreiter & Schmidhuber, 1997) and the reset/update gates of the GRU (Cho et al., 2014, *Learning Phrase Representations using RNN Encoder-Decoder for Statistical Machine Translation*) |
| `Highway` | `g * T(x) + (1 - g) * x` -- a learned interpolation between a transform and the identity | Highway Networks (Srivastava, Greff & Schmidhuber, 2015), built explicitly to let very deep plain networks train at all -- the direct ancestor of the ResNet residual connection, which is a Highway with the gate fixed at 0.5 |
| `MoE` | `sum_i softmax(Wx)_i * expert_i(x)` -- N gates routing between N experts | Jacobs & Jordan, 1991, *Adaptive Mixtures of Local Experts* -- the **dense** version trained in §1.3 below: all N experts run, every time. Modern **sparse** MoE (Shazeer et al., 2017, *Outrageously Large Neural Networks*; the Switch Transformer, Fedus et al., 2021) routes each token to only `k` of thousands of experts -- capacity scales with N, compute scales with k. §1.3 trains that version (`vectormesh`’s `SparseMoE`) next to the dense one, and §1.5 times it, because skipping the unchosen experts is what makes production MoE fast at scale |

Read top to bottom, each row is the row above with more capacity to choose from: one
gate choosing between a transform and nothing (`Gate`), one gate choosing between a
transform and the identity (`Highway`), N gates choosing between N transforms (`MoE`).

All three leave the shape alone: a `(batch, hidden)` tensor goes in and a
`(batch, hidden)` tensor comes out. What differs is what happens inside: the gate scales
each of the `hidden` features by a number between 0 and 1, the highway mixes the block's
output with its own input, and the mixture of experts chooses between several blocks, or
blends them.

This lesson's electricity-demand series has real daily and weekly structure -- exactly
the kind of thing a router could plausibly use. Predict before you look at the table:
does a mechanism that decides per window earn its parameters against a model that spends
the same parameters on a plain layer?

In [ ]:
from dataclasses import dataclass
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from loguru import logger
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from vectormesh.components import (
    Gate,
    Highway,
    MoE,
    NeuralNet,
    SparseMoE,
    load_balancing_loss,
)
from vectormesh.components.metrics import MAE
from vectormesh.training import Trainer, TrainerSettings

from australian_electricity import load_ausalia
from windowing import window_index
from notebooktester import param

# the Trainer logs every epoch at INFO; keep warnings only
logger.remove()
logger.add(sys.stderr, level="WARNING")

DAY = 48  # half-hourly steps per day
# four days of history, one day of horizon: the window 6_uncertainty/01_probabilistic
# uses, so results sit next to that notebook's
W, H = 4 * DAY, DAY
HIDDEN = 64
STRIDE = 24
SEEDS = (0, 1, 2, 3, 4)
# early stopping: training ends after PATIENCE epochs without a lower val loss, and
# the weights of the best epoch are kept. Val MAE moves up and down from one epoch to
# the next by more than the gaps between models (see the curve in 1.2); a patience of
# 10 epochs rides out that bouncing instead of stopping at the first bump.
PATIENCE = 10
MAX_EPOCHS = param(200, test=3)  # a ceiling; early stopping ends every run before it

ausalia = load_ausalia()
mu = ausalia.values[ausalia.splits["train"]].mean()
sigma = ausalia.values[ausalia.splits["train"]].std()


def windows(split: str, stride: int):
    values = (ausalia.values[ausalia.splits[split]] - mu) / sigma
    win = values[window_index(len(values), W + H, stride)]
    return win[:, :W], win[:, W:]


xtr, ytr = windows("train", STRIDE)
xva, yva = windows("val", STRIDE * 2)
xte, yte = windows("test", STRIDE * 2)
print(f"train {tuple(xtr.shape)}   val {tuple(xva.shape)}   test {tuple(xte.shape)}")

# one batch covering the whole val split -- every Trainer below shares this loader
validloader = DataLoader(TensorDataset(xva, yva), batch_size=len(xva))


def mae(y: torch.Tensor, yhat: torch.Tensor) -> float:
    return float((y - yhat).abs().mean())


def make_model(
    mechanism: str, hidden: int = HIDDEN, n_experts: int = 4, k: int = 1
) -> nn.Module:
    if mechanism == "plain":
        return NeuralNet(hidden_size=[W, hidden], out_size=H)
    if mechanism == "deeper":
        # capacity-matched control: one more Linear -> GELU than "plain", no gate at all.
        # Gate's own weight matrix costs exactly what this second layer costs at this width.
        return NeuralNet(hidden_size=[W, hidden, hidden], out_size=H)
    inp, act, out = nn.Linear(W, hidden), nn.GELU(), nn.Linear(hidden, H)
    if mechanism == "gate":
        mid = Gate(hidden_size=hidden)
    elif mechanism == "highway":
        mid = Highway(transform=NeuralNet(hidden, hidden), hidden_size=hidden)
    elif mechanism == "moe":
        mid = MoE(
            experts=[nn.Linear(hidden, hidden) for _ in range(n_experts)],
            hidden_size=hidden,
            out_size=hidden,
        )
    elif mechanism == "sparse_moe":
        mid = SparseMoE(
            experts=[nn.Linear(hidden, hidden) for _ in range(n_experts)],
            hidden_size=hidden,
            out_size=hidden,
            k=k,
        )
    else:
        raise ValueError(mechanism)
    return nn.Sequential(inp, act, mid, out)


@dataclass
class Run:
    """One training run with early stopping: the val MAE of the best epoch, the test MAE
    of the same weights, that epoch, the last epoch that ran, and the model holding the
    best epoch's weights."""

    mae: float
    test_mae: float
    best_epoch: int
    stopped_epoch: int
    model: nn.Module


def run(
    mechanism: str,
    seed: int,
    max_epochs: int,
    lr: float = 1e-3,
    batch_size: int = 64,
    balance: float = 0.0,
    **model_kwargs,
) -> Run:
    """`balance` > 0 adds that multiple of the load-balancing loss of every SparseMoE to
    the training loss (1.4); the val loss that early stopping watches never has it."""
    torch.manual_seed(seed)
    model = make_model(mechanism, **model_kwargs)
    trainloader = DataLoader(
        TensorDataset(xtr, ytr), batch_size=batch_size, shuffle=True
    )
    settings = TrainerSettings(
        epochs=max_epochs,
        metrics=[MAE()],
        logdir=Path("modellogs").resolve(),
        train_steps=len(trainloader),
        valid_steps=len(validloader),
        optimizer_kwargs={"lr": lr},
        earlystop_kwargs={"patience": PATIENCE, "save": False, "verbose": False},
    )

    def balanced_step(model: nn.Module, x: torch.Tensor, y: torch.Tensor):
        loss = nn.functional.mse_loss(model(x), y)
        if model.training:
            loss = loss + balance * load_balancing_loss(model)
        return loss

    trainer = Trainer(
        model=model,
        settings=settings,
        loss_fn=nn.MSELoss(),
        optimizer=torch.optim.Adam,
        traindataloader=trainloader,
        validdataloader=validloader,
        step=balanced_step if balance else None,
        progress=False,
    )
    # the result and trainer.model describe the epoch with the lowest val loss.
    # TrainResult counts epochs from 0; this notebook counts from 1, as the curve does.
    result = trainer.loop()
    trainer.model.eval()
    with torch.no_grad():
        test_mae = mae(yte, trainer.model(xte))
    return Run(
        mae=result.metric_dict["MAE"],
        test_mae=test_mae,
        best_epoch=result.epoch + 1,
        stopped_epoch=result.stopped_epoch + 1,
        model=trainer.model,
    )


def mae_curve(
    mechanism: str,
    epochs: int,
    seed: int = 0,
    lr: float = 1e-3,
    batch_size: int = 64,
    **model_kwargs,
) -> tuple[list[float], list[float], list[float]]:
    """The same training as `run`, without early stopping, for a fixed number of epochs.
    Returns train MAE, val MAE and val loss (MSE) after every epoch: a Reporter reads val
    MAE and val loss from the Trainer, and does one extra forward pass over train for
    the train-side curve."""
    torch.manual_seed(seed)
    model = make_model(mechanism, **model_kwargs)
    trainloader = DataLoader(
        TensorDataset(xtr, ytr), batch_size=batch_size, shuffle=True
    )
    train_curve: list[float] = []
    val_curve: list[float] = []
    val_loss: list[float] = []

    def track(epoch, train_loss, test_loss, metric_dict):
        val_curve.append(metric_dict["MAE"])
        val_loss.append(test_loss)
        with torch.no_grad():
            train_curve.append(mae(ytr, model(xtr)))

    settings = TrainerSettings(
        epochs=epochs,
        metrics=[MAE()],
        logdir=Path("modellogs").resolve(),
        train_steps=len(trainloader),
        valid_steps=len(validloader),
        optimizer_kwargs={"lr": lr},
        earlystop_kwargs=None,
    )
    trainer = Trainer(
        model=model,
        settings=settings,
        loss_fn=nn.MSELoss(),
        optimizer=torch.optim.Adam,
        traindataloader=trainloader,
        validdataloader=validloader,
        reporters=[track],
        progress=False,
    )
    trainer.loop()
    return train_curve, val_curve, val_loss


def early_stop_epochs(
    val_loss: list[float], patience: int = PATIENCE
) -> tuple[int, int]:
    """Where early stopping would pick and stop on a recorded val-loss curve: the epoch
    with the lowest loss so far, and the epoch `patience` epochs without improvement
    later (or the last epoch). The same rule as the Trainer's EarlyStopping."""
    best, best_epoch, waited = float("inf"), 0, 0
    for epoch, loss in enumerate(val_loss, start=1):
        if loss < best:
            best, best_epoch, waited = loss, epoch, 0
        else:
            waited += 1
            if waited >= patience:
                return best_epoch, epoch
    return best_epoch, len(val_loss)

## 1.2 When to stop: the curve, and early stopping

Train too briefly and a model is still learning; train too long and it starts fitting
the train windows at the cost of the val windows. You met the standard answer in lesson 1
(`1_representation_and_head/03_training`): **early stopping**. After every epoch the
Trainer computes the val loss; once it has not dropped for `PATIENCE` epochs in a row,
training ends and the weights of the epoch with the lowest val loss are kept.
`MAX_EPOCHS` is then only a ceiling, not a choice.

To see what that rule does, the cell below trains `gate` and its capacity-matched
control `deeper` (seed 0) for a fixed 90 epochs with early stopping off, so the whole
curve is visible, and marks where early stopping with `PATIENCE = 10` would pick its
epoch (dot) and stop (dashed line). Early stopping watches the val *loss*, the MSE the
model trains on, not the val MAE plotted here; the two mostly move together, not
always.

Before you run it: where do you expect the val curves to flatten, and do you expect
train to pull away from val?

In [ ]:
CURVE_EPOCHS = param(90, test=8)

fig, ax = plt.subplots(figsize=(9, 3.2))
colors = {"deeper": "C0", "gate": "C1"}
for name in ("deeper", "gate"):
    train_curve, val_curve, val_loss = mae_curve(name, epochs=CURVE_EPOCHS)
    best_epoch, stop_epoch = early_stop_epochs(val_loss)
    epochs_axis = range(1, len(val_curve) + 1)
    ax.plot(epochs_axis, train_curve, color=colors[name], lw=0.8, ls=":", alpha=0.6)
    ax.plot(epochs_axis, val_curve, color=colors[name], lw=1.4, label=name)
    ax.axvline(stop_epoch, color=colors[name], lw=1, ls="--")
    ax.plot(best_epoch, val_curve[best_epoch - 1], "o", color=colors[name])
    print(
        f"{name:7s} best epoch {best_epoch:3d} (val MAE {val_curve[best_epoch - 1]:.4f})"
        f"   early stopping stops at epoch {stop_epoch:3d}"
        f"   val MAE at epoch {len(val_curve)}: {val_curve[-1]:.4f}"
    )
ax.set_title(
    f"Seed 0: dot = best val loss, dashed = where early stopping "
    f"(patience {PATIENCE}) stops"
)
ax.set_xlabel("epoch")
ax.set_ylabel("MAE  (dotted = train, solid = val)")
ax.legend(fontsize=8)
plt.tight_layout()

Both val curves drop steeply over the first 20 epochs, bouncing as they go, then flatten
into a noisy plateau between about 0.15 and 0.17 that lasts to epoch 90. Train (dotted)
sits about 0.03 below val, and that gap does not widen over the plateau: a stable
generalisation gap, not runaway overfitting.

Early stopping picks a low point on that plateau: epoch 35 for `deeper` (val MAE 0.1552)
and epoch 49 for `gate` (0.1488), and stops 10 epochs later, at 45 and 59. Training on
to epoch 90 buys nothing: `deeper` ends at the same 0.1552, `gate` at 0.1597, worse
than its chosen epoch. The dot is the lowest val *loss* up to the stop, so on this noisy
plateau the MAE curve can still dip below it once by chance (`deeper` near epoch 59);
that bouncing is also why `PATIENCE` is 10 and not 2.

Every run below uses this rule: the same `PATIENCE`, the same ceiling `MAX_EPOCHS`, so
each model trains as long as its own val loss keeps improving, and no model gets an
epoch count picked by hand. For seed 0, `run()` picks the same best epochs as the dots
above (35 for `deeper`, 49 for `gate`): the curve and the runs are the same training.

One caveat holds for every row of the val column: the stop epoch is chosen on val and the
MAE is reported on val, so each val number is slightly optimistic, equally for every model.
Every run is therefore also scored once, with the weights of its best epoch, on the test
windows (the last full year of the series), which take part in no decision.

## 1.3 The full family, trained with early stopping

In [ ]:
# (label, mechanism, make_model/run kwargs) -- `deeper (hidden 96)` is the control for
# the 28k-32k-parameter models (highway, moe, sparse_moe N=4): 32,496 parameters, no
# mechanism. sparse_moe appears twice: once at the
# same N as dense moe (does routing beat blending at matched capacity?), once at 16x
# the experts (does the extra capacity a router can afford almost for free help?).
MODEL_CONFIGS = [
    ("plain", "plain", {}),
    ("deeper", "deeper", {}),
    ("deeper (hidden 96)", "deeper", {"hidden": 96}),
    ("gate", "gate", {}),
    ("highway", "highway", {}),
    ("moe", "moe", {"n_experts": 4}),
    ("sparse_moe (N=4)", "sparse_moe", {"n_experts": 4, "k": 1}),
    ("sparse_moe (N=64)", "sparse_moe", {"n_experts": 64, "k": 1}),
]

results: dict[str, dict] = {}
runs: dict[str, list[Run]] = {}
for label, mechanism, kwargs in MODEL_CONFIGS:
    runs[label] = [run(mechanism, seed, MAX_EPOCHS, **kwargs) for seed in SEEDS]
    scores = [r.mae for r in runs[label]]
    tests = [r.test_mae for r in runs[label]]
    params = sum(p.numel() for p in make_model(mechanism, **kwargs).parameters())
    results[label] = {
        "mae": float(np.mean(scores)),
        "std": float(np.std(scores)),
        "test": float(np.mean(tests)),
        "test_std": float(np.std(tests)),
        "best_epoch": float(np.mean([r.best_epoch for r in runs[label]])),
        "stopped_epoch": float(np.mean([r.stopped_epoch for r in runs[label]])),
        "params": params,
    }
    print(
        f"{label:20s} val {results[label]['mae']:.5f} +/- {results[label]['std']:.5f}   "
        f"test {results[label]['test']:.5f} +/- {results[label]['test_std']:.5f}   "
        f"best epochs {[r.best_epoch for r in runs[label]]}   {params:>8,} params"
    )

## 1.4 The table

In [ ]:
order = sorted(results, key=lambda k: results[k]["test"])
print(
    f"{'model':20s} {'test MAE':>9} {'+/-':>7} {'val MAE':>9} {'+/-':>7} "
    f"{'best ep':>8} {'stop ep':>8} {'params':>9}"
)
print("-" * 84)
for name in order:
    r = results[name]
    print(
        f"{name:20s} {r['test']:9.5f} {r['test_std']:7.5f} {r['mae']:9.5f} "
        f"{r['std']:7.5f} {r['best_epoch']:8.1f} {r['stopped_epoch']:8.1f} "
        f"{r['params']:9,}"
    )

Read every gap against the spread: the std over 5 seeds runs from 0.002 to 0.012. Early
stopping chose each run's epoch on the val windows, so the val column is slightly
optimistic for every row; the test windows took part in no decision, so the **test column**
is the one to quote. The two columns do not rank the models the same way, which is itself a
sign of how little separates them.

- **`gate` against `deeper`, the same 19,632 params:** test 0.1672 ± 0.0104 against
  0.1655 ± 0.0088 (val 0.1593 against 0.1605). The gap is a tenth of either std and changes
  sign between the columns: no sign that the gate does anything a second plain layer does
  not.
- **`highway` and dense `moe` have the lowest test MAE** (0.1629 ± 0.0038 and
  0.1625 ± 0.0027), at 1.8x and 2.1x `plain`'s parameters. Against `plain`
  (0.1683 ± 0.0035) the gap of 0.005 is about 1.5 std. But `deeper (hidden 96)`, a model
  with their parameter count (32,496) and no mechanism at all, scores 0.1634 ± 0.0019. The
  gain over `plain` is what twice the parameters buys; the mechanism adds nothing that a
  plain layer of the same size does not. That is the capacity-matched control doing its
  job.
- **`sparse_moe (N=4)` has exactly dense `moe`'s 32,372 params** and scores
  0.1692 ± 0.0061 against 0.1625 ± 0.0027. The gap of 0.007 is one to two std: no sign that
  routing beats blending, and no firm sign that it loses. The two use their parameters
  differently: the dense mixture blends all four experts, the sparse one sends each window
  through one expert and scales its output by the router's probability.
- **`sparse_moe (N=64)`, at 285,872 params (18.5x `plain`),** scores 0.1661 ± 0.0031, no
  better than `deeper` with 19,632 (0.1655): fifteen times the parameters for nothing. The
  next section looks at what the router does with those 64 experts.

On this series nothing earns its parameters against a plain layer of the same size.

The best and stop columns show that early stopping ended every run long before the ceiling
of 200 epochs, and the stop epoch is always the best epoch plus `PATIENCE`.

### Where do the 64 experts' windows go?

A top-1 router sends each window to one expert. If it sends nearly all windows to a few
experts, the rest are parameters that never see data; if it sends all of them to one, the
model has turned into a single expert and the other 63 are dead weight. That is **router
collapse**. Two things could produce it: the router starts out concentrated (its weights
are random), or training concentrates it further.

The cell below takes each seed's trained `sparse_moe (N=64)` from 1.3, counts how many of
the 361 val windows its router sends to each expert, does the same for the untrained model
with the same seed, and measures how far the router's weights moved in training. Predict
first: are more or fewer experts in use after training than before?

In [ ]:
N_ROUTED = 64


def expert_counts(model: nn.Module) -> torch.Tensor:
    """How many val windows the router of a trained Linear -> GELU -> SparseMoE -> Linear
    model sends to each expert. SparseMoE routes each row to the argmax of
    `router(h)` (top-1), where h is the hidden vector after the first Linear -> GELU."""
    inp, act, moe = model[0], model[1], model[2]
    with torch.no_grad():
        chosen = moe.router(act(inp(xva))).argmax(dim=-1)
    return torch.bincount(chosen, minlength=N_ROUTED)


def describe(counts: torch.Tensor) -> str:
    top = counts.sort(descending=True).values
    return (
        f"{int((counts > 0).sum()):2d} of {N_ROUTED} experts used   "
        f"top expert {int(top[0]):3d} of {len(xva)} windows   "
        f"top five {int(top[:5].sum()):3d}"
    )


for seed, r in zip(SEEDS, runs["sparse_moe (N=64)"]):
    trained = r.model
    torch.manual_seed(seed)
    untrained = make_model("sparse_moe", n_experts=N_ROUTED, k=1)
    router_change = (trained[2].router.weight - untrained[2].router.weight).abs().max()
    print(f"seed {seed}  untrained: {describe(expert_counts(untrained))}")
    print(f"        trained:   {describe(expert_counts(trained))}")
    print(
        f"        largest change in a router weight during training: {router_change:.2e}"
    )

**Usage starts concentrated and training collapses it completely.** The untrained routers
send the val windows to 9 to 16 of the 64 experts, and their five most-used experts take
316 to 343 of the 361 windows. After training every seed uses **one** expert, which takes
all 361 windows. The router weights moved (the largest change is 0.26 to 0.69), so this is
router collapse learned during training, on top of an uneven start.

That fits the table: the 64-expert model scores like `deeper` (0.1661 against 0.1655 on
test), because a router that sends every window to one expert makes a network with a single
expert, its output scaled by the router's probability. The 285,872 parameters are 63
experts that no window reaches. Why the router collapses is not tested here; the usual
account is a feedback loop, in which an expert that receives a few more windows trains more,
becomes better, and is preferred by the router. What the next cell does test is the standard
remedy.

### What a balancing loss does

The Switch Transformer adds an auxiliary loss to the training loss. It is 1 when the
windows are spread evenly over the experts and grows as the router concentrates
(`load_balancing_loss`: `N · Σ f_i · P_i`, with `f_i` the fraction of windows sent to
expert `i` and `P_i` the router's mean probability for it), weighted by a small factor. The
cell trains `sparse_moe (N=64)` again with three weights. The loss is added only while
training, so the val loss that early stopping watches is the same MSE as before. Predict
first: with a loss that pushes the usage towards many experts, what happens to the MAE?

In [ ]:
# 1e-2 is the weight Fedus et al. (2021) use for the load-balancing loss
BALANCES = (1e-4, 1e-3, 1e-2)

balanced_runs = {
    b: [
        run("sparse_moe", seed, MAX_EPOCHS, balance=b, n_experts=N_ROUTED, k=1)
        for seed in SEEDS
    ]
    for b in BALANCES
}
print(f"{'':32s} {'val MAE':>17}   {'test MAE':>17}   experts used on val, per seed")
for label, rs in [
    ("sparse_moe (N=64), no balance", runs["sparse_moe (N=64)"]),
    *[(f"... + balance {b:g}", balanced_runs[b]) for b in BALANCES],
]:
    v = np.array([r.mae for r in rs])
    t = np.array([r.test_mae for r in rs])
    used = [int((expert_counts(r.model) > 0).sum()) for r in rs]
    print(
        f"{label:32s} {v.mean():.4f} +/- {v.std():.4f}   "
        f"{t.mean():.4f} +/- {t.std():.4f}   {used}"
    )

Balancing does what it is for: a weight of 1e-4 changes nothing (one expert in four seeds,
two in the fifth), 1e-3 spreads the windows over 4 to 6 experts and 1e-2 over 13 to 19,
never anywhere near 64. But the MAE gets worse as the usage gets wider: on test 0.1661 with
one expert, 0.1893 with 4 to 6 and 0.2187 with 13 to 19, each step far outside the spread
(0.003 to 0.009). On this series, using more experts costs accuracy, and the collapsed
model is the best of the 64-expert variants.

A balanced router is a router that satisfies the balancing loss, not one that forecasts
better. Whether a series has regimes that separate experts can divide up is what decides
whether spreading the load pays, and this one, at this size, does not reward it. Usage
is a diagnosis (is the router collapsed?), not a goal.

## 1.5 Is sparse routing cheap to scale?

1.4 answered "does it forecast better": no, and the router collapses. This section asks a
separate question: does the mechanism deliver its production promise, holding far more
capacity than any single input pays for in compute? `MoE` is *dense*: every expert runs on
every input, so more experts always cost more compute. `SparseMoE` gathers the windows of
each expert and runs only the experts that received any: an expert with no window does no
work.

The test: fix `k=1` and a batch of 64 of this notebook's own windows, grow `N` from 4 to
256, and time one forward pass of the block. Three arms: dense `MoE`, the control, which has
no reason to stay flat; `SparseMoE` with its own freshly initialised router, which reports
how many experts it reaches (idle experts cost nothing); and `SparseMoE` with an even
router that spreads the 64 windows perfectly over the experts, window `i` to expert
`i mod N`, the load a balancing loss aims for. Predict first: which arm grows slowest with
`N`, and which is fastest at `N=256`?

In [ ]:
import time

from vectormesh.components import SparseMoE

N_EXPERTS_TO_CHECK = param((4, 16, 64, 256), test=(2, 4))
ROUTING_BATCH = 64
REPS = param(30, test=3)

# a real batch of this notebook's own windows, projected to HIDDEN once -- the
# component under test is the mixture-of-experts block, not the input projection
probe_input = nn.Linear(W, HIDDEN)
probe_x = probe_input(xtr[:ROUTING_BATCH])


def time_forward(model: nn.Module, x: torch.Tensor, reps: int) -> float:
    for _ in range(5):
        model(x)
    start = time.perf_counter()
    for _ in range(reps):
        model(x)
    return (time.perf_counter() - start) / reps * 1000


class EvenRouter(nn.Module):
    """A router that spreads a batch over the experts perfectly evenly (window i goes to
    expert i mod N), whatever the input: the fastest load a sparse layer can get."""

    def __init__(self, n_experts: int):
        super().__init__()
        self.n_experts = n_experts

    def forward(self, h: torch.Tensor) -> torch.Tensor:
        chosen = torch.arange(len(h)) % self.n_experts
        return 10.0 * nn.functional.one_hot(chosen, self.n_experts).float()


sparse_timings: dict[int, dict] = {}
even_timings: dict[int, dict] = {}
dense_timings: dict[int, dict] = {}
for n_experts in N_EXPERTS_TO_CHECK:
    torch.manual_seed(0)
    sparse = SparseMoE(
        experts=[nn.Linear(HIDDEN, HIDDEN) for _ in range(n_experts)],
        hidden_size=HIDDEN,
        out_size=HIDDEN,
        k=1,
    )
    torch.manual_seed(0)
    dense = MoE(
        experts=[nn.Linear(HIDDEN, HIDDEN) for _ in range(n_experts)],
        hidden_size=HIDDEN,
        out_size=HIDDEN,
    )

    even = SparseMoE(
        experts=[nn.Linear(HIDDEN, HIDDEN) for _ in range(n_experts)],
        hidden_size=HIDDEN,
        out_size=HIDDEN,
        k=1,
    )
    even.router = EvenRouter(n_experts)

    with torch.no_grad():
        n_used = int(torch.bincount(sparse.router(probe_x).argmax(-1)).count_nonzero())
    sparse_ms = time_forward(sparse, probe_x, REPS)
    even_ms = time_forward(even, probe_x, REPS)
    dense_ms = time_forward(dense, probe_x, REPS)
    params = sum(p.numel() for p in sparse.parameters())  # identical count, both sides

    sparse_timings[n_experts] = {"ms": sparse_ms, "params": params}
    even_timings[n_experts] = {"ms": even_ms, "params": params}
    dense_timings[n_experts] = {"ms": dense_ms, "params": params}
    print(
        f"N={n_experts:4d} experts   sparse {sparse_ms:7.3f} ms ({n_used:3d} experts "
        f"reached)   sparse, even load {even_ms:7.3f} ms   dense {dense_ms:7.3f} ms   "
        f"{params:>9,} params"
    )
first, last = min(N_EXPERTS_TO_CHECK), max(N_EXPERTS_TO_CHECK)
for name, timings in [
    ("sparse", sparse_timings),
    ("sparse, even load", even_timings),
    ("dense", dense_timings),
]:
    print(
        f"{name:18s} time grows {timings[last]['ms'] / timings[first]['ms']:1.1f}x "
        f"from N={first} to N={last}   (parameters grow "
        f"{timings[last]['params'] / timings[first]['params']:.0f}x)"
    )

Measured on a CPU, with a batch of 64 and `hidden=64`. Absolute times vary with the machine
and the run; read the shape.

- **Dense** grows about 16 to 17x for 64x the parameters, close to proportional to `N`,
  since every expert runs on every window.
- **Sparse with its own router** grows about 3x, because its router reaches only 4 to 12
  experts however large `N` is. It is cheap because most of its experts do nothing: the
  concentration seen in 1.4, read as a speed-up.
- **Sparse with an even load** grows about 10x and flattens between `N=64` and `N=256`: a
  batch of 64 windows cannot occupy more than 64 experts. The time follows the number of
  experts that receive a window, each paying the gather and scatter, not the parameter
  count.
- **At every `N` measured, sparse is slower than dense**: about two times at `N=256` with
  its own router and about six times with an even load. With experts this small and a batch
  of 64, running every expert on every window is cheaper than launching one expert per few
  windows.

So compute does decouple from parameters (10x for 64x with an even load), but only once the
work inside an expert outweighs the cost of launching it. At this size, on this CPU, it does
not, and "does it forecast better" (no) and "is it cheap to scale" (not here) have different
answers.

`04_search` is where you stop checking one hand-picked configuration at a time and let
a search cover the space properly.

## What to take from this

- Gate, Highway and MoE are one idea -- a learned keep-or-replace decision -- at
  increasing resolution, and each one traces to a specific paper rather than floating
  free in a library's component list.
- The mechanism is general-purpose. It showed up here because this lesson had a window
  and a baseline ready, not because time series needs it more than any other modality.
- **Look at the curve, then let early stopping pick the epoch.** 1.2 shows the train/val
  curve and where early stopping stops on it; 1.3 trains every model with that same rule,
  so no model gets a hand-picked epoch count, and the table reports when each one stopped
  and the test MAE of its chosen epoch.
- **A control with the same parameters decides.** `gate` ties `deeper`, and the gain of
  `highway` and `moe` over `plain` is also reached by `deeper (hidden 96)`, a plain layer
  with their parameter count. "Capacity-matched control" is doing the real work in that
  sentence, not "gate", "highway" or "sparse_moe" by name.
- **Parameters trained and parameters used are different numbers for a router, and a
  router can stop using most of them.** After training, the 64-expert router sends every
  window to one expert; a balancing loss spreads the windows and costs accuracy. Usage is
  a diagnosis, not a goal.
- **"Does it forecast better" and "is it cheap to scale" are different questions.** 1.3
  and 1.4 answer the first, 1.5 the second, and sparse stays slower than dense at every
  size measured on this CPU.

## Exercises

**E1 -- predict, then measure.** Before re-running this notebook, settle on a ranking you
expect for all eight trained models, and say whether you expect this series' structure to
change the 1.4 verdict that `gate` does no better than `deeper`. Check both.

**E2 -- turn the gate off.** Force `Gate`'s output to equal its input (patch the forward
pass, or initialise its linear layer to a large positive bias so the sigmoid saturates near
1) and confirm the block's score converges toward `deeper`'s -- the collapse argument from
`1_representation_and_head/02_nonlinearity` §2.1, for a gate instead of a stacked linear
layer.

**E3 -- more experts, dense.** Re-run `make_model("moe")` with 2, 4, 8 and 16 experts, each
trained with the same early stopping as 1.3, and compare against `deeper (hidden 96)` and
`sparse_moe (N=64)`'s rows. Does the dense mixture find anything to split on as `N` grows, or
does the score just track the parameter count?

**E4 -- a different window.** Swap in `pedestrian_counts` (Monash `monash_tsf`, config
`pedestrian_counts`; `australian_electricity.py` shows how to load a config and cut it in
time) and re-run the whole notebook. Does the ranking change on a series with an even
sharper daily cycle?

**E5 -- `k>1`.** Re-run the router cell of 1.4 and the timing sweep of 1.5 with `k=2` and
`k=4` (`normalize=True` is allowed from `k=2`). Predict first: does the number of experts in
use after training grow with `k`, and does sparse time come closer to dense or move further
away?

**E6 -- a weight between 1e-3 and 1e-2.** 1.4 swept the balancing weight over three
values. Add 3e-3 and 3e-2 and compare every row against `deeper`. Is there a weight at which
the router spreads its load and the MAE is no worse than the collapsed model's?